In [ ]:

import base64, gzip, hashlib, json, os, pathlib, signal, subprocess, sys, time
session_started = time.monotonic()
import torch
CAP = 13000.0
work = pathlib.Path("/kaggle/working/burgers"); work.mkdir(parents=True, exist_ok=True)
payload = {"pikan_campaign.py": ('...', False), "burgers_coleHopf_1025x129.npz": ('...', True)}
hashes = {}
for fname, (blob, gz) in payload.items():
    raw = base64.b64decode(blob)
    raw = gzip.decompress(raw) if gz else raw
    (work / fname).write_bytes(raw)
    hashes[fname] = hashlib.sha256(raw).hexdigest()
gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print(json.dumps({"gpus": gpus, "torch": torch.__version__, "hashes": hashes, "cap": CAP}, indent=1), flush=True)
if not gpus or any("T4" not in g.upper() for g in gpus):
    raise RuntimeError(f"Expected T4 GPUs, got {gpus!r}; CPU fallback disabled.")
subs = {"@BU": str(work / "burgers_coleHopf_1025x129.npz"), "@OUT": str(work / "results")}
cli = ['campaign', '--problem', 'burgers', '--ref', '@BU', '--out', '@OUT/burgers', '--arms', 'mlp', 'rbf@16', 'rbf@48', '--seeds', '301', '302', '303', '304', '305', '--adam-steps', '20000', '--lbfgs-seconds', '600', '--wall-cap', '10000']
for k, v in subs.items():
    cli = [c.replace(k, v) if c.startswith(k) or k in c else c for c in cli]
log = (work / "runner.log").open("w")
proc = subprocess.Popen([sys.executable, str(work / "pikan_campaign.py"), *cli], stdout=log,
                        stderr=subprocess.STDOUT, start_new_session=True, cwd=str(work))
timed_out = False
while proc.poll() is None:
    if time.monotonic() - session_started > CAP:
        timed_out = True
        os.killpg(proc.pid, signal.SIGTERM)
        time.sleep(10)
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        break
    time.sleep(2)
rc = proc.wait()
summary = {"status": "timeout_stopped" if timed_out else ("complete" if rc == 0 else "runner_failed"),
           "return_code": rc, "wall_seconds": time.monotonic() - session_started, "cap_seconds": CAP,
           "gpus": gpus, "hashes": hashes, "scientific_accepted": False, "reference_accepted": False}
(work / "session_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, ))
print((work / "runner.log").read_text()[-3000:])
